# Australian Law LLM — Colab Training Notebook

Train a larger model on a free or paid Colab GPU, then download the adapters to run locally.

| Runtime | VRAM | Recommended config | Cost |
|---------|------|-------------------|------|
| T4 (free) | 16 GB | `--gpu 16gb` → Llama-3.2-3B | Free (session limits apply) |
| A100 (Pro) | 40 GB | `--gpu 24gb` → Llama-3.1-8B | ~$10/month |
| L4 (Pro+) | 24 GB | `--gpu 24gb` → Llama-3.1-8B | ~$50/month |

**Before running:** set your runtime to GPU via `Runtime → Change runtime type → T4 GPU`

**Output:** LoRA adapter files that you download and drop into your local `Australian-Law-LLM/` folder, then serve with `python serve.py`.

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip())

import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
print(f"\nVRAM: {vram:.1f} GB")

if vram >= 35:
    GPU_TIER = '24gb'
    print("→ Using --gpu 24gb (Llama-3.1-8B, seq=2048)")
elif vram >= 12:
    GPU_TIER = '16gb'
    print("→ Using --gpu 16gb (Llama-3.2-3B, seq=1024)")
else:
    GPU_TIER = '8gb'
    print("→ Using --gpu 8gb  (Llama-3.2-3B, seq=512)")

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# Order matters: torch first, then unsloth, then pin transformers, then the rest.
# Colab already has torch with CUDA so we skip that step.

import sys

print("Installing Unsloth...")
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("Pinning transformers...")
!pip install -q transformers==4.56.1

print("Installing remaining dependencies...")
!pip install -q \
    trl>=0.11.0 \
    peft>=0.13.0 \
    accelerate>=0.34.0 \
    bitsandbytes>=0.44.0 \
    torchao==0.13.0 \
    "unsloth_zoo==2026.6.1" \
    datasets>=3.0.0 \
    kagglehub>=0.3.0 \
    hf_transfer

print("\nDone. Restarting runtime to apply package changes...")

# Restart so the newly installed packages are picked up cleanly
import os
os.kill(os.getpid(), 9)

In [ ]:
# ── Cell 3: Re-detect GPU after restart ───────────────────────────────────────
import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram:.1f} GB")

if vram >= 35:
    GPU_TIER = '24gb'
elif vram >= 12:
    GPU_TIER = '16gb'
else:
    GPU_TIER = '8gb'

print(f"GPU tier: --gpu {GPU_TIER}")

In [ ]:
# ── Cell 4: Clone the repo ────────────────────────────────────────────────────
import os

REPO_DIR = '/content/Australian-Law-LLM'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mcrolfey/Australian-Law-LLM.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Cell 5: Kaggle credentials ────────────────────────────────────────────────
# Upload your kaggle.json using the Colab file browser (left sidebar → Files → Upload),
# or paste your username and key below.
#
# Option A — paste credentials directly (easier)
KAGGLE_USERNAME = ""   # ← your Kaggle username
KAGGLE_KEY      = ""   # ← your Kaggle API key

# Option B — upload kaggle.json via the file browser then run this cell
# (leave the variables above empty and it will look for the uploaded file)

import os, json

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

if KAGGLE_USERNAME and KAGGLE_KEY:
    with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
    print("Kaggle credentials written.")
elif os.path.exists('/content/kaggle.json'):
    import shutil
    shutil.copy('/content/kaggle.json', f'{kaggle_dir}/kaggle.json')
    os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
    print("kaggle.json copied from upload.")
elif os.path.exists(f'{kaggle_dir}/kaggle.json'):
    print("kaggle.json already present.")
else:
    print("ERROR: No Kaggle credentials found. Fill in KAGGLE_USERNAME/KEY above or upload kaggle.json.")

In [ ]:
# ── Cell 6 (Optional): Step 1 — Continued Pre-Training (CPT) ─────────────────
#
# CPT reads raw legal text and injects factual knowledge before SFT.
# Skip this cell if you only want to run SFT.
#
# On a T4 this takes ~2–3 hours for 500 steps at seq=1024.
# On an A100 this takes ~45 min for 500 steps at seq=8192.

RUN_CPT = True   # ← set to False to skip

if RUN_CPT:
    !python cpt_train.py --gpu {GPU_TIER} --steps 500
else:
    print("CPT skipped.")

In [ ]:
# ── Cell 7: Step 2 — SFT Instruction Fine-Tuning ─────────────────────────────
#
# Trains the model to answer Australian law questions.
# If CPT was run above, pass --cpt-model to start from domain-adapted weights.

import os

CPT_MODEL_DIR = 'lora_cpt_law_model'
SFT_STEPS = 200   # increase for better results; 60 is the quick-test default

if os.path.isdir(CPT_MODEL_DIR):
    print(f"CPT adapters found at {CPT_MODEL_DIR} — using as SFT base.")
    !python train.py --gpu {GPU_TIER} --steps {SFT_STEPS} --cpt-model {CPT_MODEL_DIR}
else:
    print("No CPT adapters found — training SFT from base model.")
    !python train.py --gpu {GPU_TIER} --steps {SFT_STEPS}

In [ ]:
# ── Cell 8: Run batch evaluation ──────────────────────────────────────────────
# Optional — runs all 100 benchmark questions and saves results.
# Takes ~15 min on a T4.

RUN_EVAL = True   # ← set to False to skip

if RUN_EVAL:
    !python batch_test.py --model lora_australian_law_model
else:
    print("Eval skipped.")

In [ ]:
# ── Cell 9: Download adapters ─────────────────────────────────────────────────
# Zips the LoRA adapter folders and downloads them to your machine.
# Unzip and drop into your local Australian-Law-LLM/ folder,
# then run: python serve.py --model lora_australian_law_model

import os
from google.colab import files

def zip_and_download(folder, zipname):
    if os.path.isdir(folder):
        !zip -r {zipname} {folder}/
        files.download(zipname)
        print(f"Downloaded {zipname}")
    else:
        print(f"Skipping {folder} — not found")

zip_and_download('lora_australian_law_model', 'lora_australian_law_model.zip')
zip_and_download('lora_cpt_law_model',        'lora_cpt_law_model.zip')
zip_and_download('batch_results_finetuned.txt', 'batch_results_finetuned.txt')

In [ ]:
# ── Cell 10 (Optional): Push adapters to Hugging Face Hub ────────────────────
# Save your adapters to a private HF repo so you can pull them on any machine
# without re-training. Much faster than re-downloading via zip.
#
# Requires a free account at huggingface.co and a write-access token.

HF_TOKEN   = ""    # ← your HuggingFace write token (hf_...)
HF_REPO_ID = ""    # ← e.g. "mcrolfey/australian-law-llm-adapters"

if HF_TOKEN and HF_REPO_ID:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)

    # Create the repo if it doesn't exist
    api.create_repo(repo_id=HF_REPO_ID, private=True, exist_ok=True)

    # Upload SFT adapters
    if os.path.isdir('lora_australian_law_model'):
        api.upload_folder(
            folder_path='lora_australian_law_model',
            repo_id=HF_REPO_ID,
            path_in_repo='lora_australian_law_model',
            token=HF_TOKEN,
        )
        print(f"SFT adapters pushed to {HF_REPO_ID}")

    # Upload CPT adapters
    if os.path.isdir('lora_cpt_law_model'):
        api.upload_folder(
            folder_path='lora_cpt_law_model',
            repo_id=HF_REPO_ID,
            path_in_repo='lora_cpt_law_model',
            token=HF_TOKEN,
        )
        print(f"CPT adapters pushed to {HF_REPO_ID}")
else:
    print("HF_TOKEN or HF_REPO_ID not set — skipping HuggingFace push.")